In [ ]:
import numpy as np
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import h5py
from torch.utils.data import Dataset, DataLoader
from tqdm import trange, tqdm


sys.path.append(os.path.abspath(os.path.join('../')))

from autoencoder import Autoencoder

In [ ]:
main_path = os.getcwd()

requires_grad = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'      # cpu or cuda

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
class KolmogorovH5InMemoryDataset(Dataset):

    def __init__(self, h5_path: str, split: str, transform=None, device=None):

        super().__init__()
        assert split in ["train", "test"], "split must be 'train' or 'test'"

        self.split = split
        self.transform = transform

        with h5py.File(h5_path, "r") as f:
            vel = f[f"velocity_{split}"][:]          
            diss = f[f"dissipation_{split}"][:] 

        vel = torch.tensor(vel, dtype=torch.float32).squeeze()
        diss = torch.tensor(diss, dtype=torch.float32)

        if device is not None:
            vel = vel.to(device)
            diss = diss.to(device)

        self.velocity = vel
        self.dissipation = diss

    def __len__(self):
        return len(self.velocity)

    def __getitem__(self, idx):
        velocity = self.velocity[idx]
        dissipation = self.dissipation[idx]

        if self.transform:
            velocity = self.transform(velocity)

        return velocity, dissipation


In [ ]:
data_file = "../datasets/phys_128x128.h5"

# Load entire dataset into memory
train_dataset = KolmogorovH5InMemoryDataset(data_file, split="train", device=device)
test_dataset = KolmogorovH5InMemoryDataset(data_file, split="test", device=device)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Linear Autoencoder (PCA/POD)

In [ ]:
z_size = 8
linear_autoencoder = Autoencoder(latent_dim=z_size, base=32, activation=nn.Identity()).to(device=device)

pytorch_total_params = sum(p.numel() for p in linear_autoencoder.parameters() if p.requires_grad)
print(f"Number of trainable parameters in the autoencoder: {pytorch_total_params:,}")


criterion = nn.MSELoss()
optimizer = optim.Adam(linear_autoencoder.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 30
pbar = tqdm(range(num_epochs))

losses = []
val_losses = []

for epoch in pbar:
  epoch_loss = 0
  for batch in train_loader:
    # Zero the gradients
    optimizer.zero_grad()
    velocity_gt, _ = batch
    # Forward pass
    output = linear_autoencoder(velocity_gt)

    # Compute the loss
    loss = criterion(output, velocity_gt)
    epoch_loss += loss/len(train_loader)

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

  losses.append(epoch_loss.item())
  print("Epoch", epoch, "Loss:", epoch_loss.item())

  with torch.no_grad():
    val_loss = 0
    for batch in test_loader:
        # Forward pass
        velocity_gt, _ = batch
        output = linear_autoencoder(velocity_gt)

        # Compute the loss
        val_loss += criterion(output, velocity_gt)/len(test_loader)
    
    val_losses.append(val_loss.item())
    print("Val Loss:", val_loss.item())


torch.save(linear_autoencoder.state_dict(), f"./pretrained_models/linear_autoencoder_{z_size}.pth")

plt.plot(losses, label="Train")
plt.plot(val_losses, label="Valid")
plt.legend()
plt.show()

# Nonlinear Autoencoder (nCAE)

In [ ]:
z_size = 8
autoencoder = Autoencoder(latent_dim=z_size, base=32, activation=nn.Tanh()).to(device=device)

pytorch_total_params = sum(p.numel() for p in autoencoder.parameters() if p.requires_grad)
print(f"Number of trainable parameters in the autoencoder: {pytorch_total_params:,}")


criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 30
pbar = tqdm(range(num_epochs))

losses = []
val_losses = []

for epoch in pbar:
  epoch_loss = 0
  for batch in train_loader:
    # Zero the gradients
    optimizer.zero_grad()
    velocity_gt, _ = batch
    # Forward pass
    output = autoencoder(velocity_gt)

    # Compute the loss
    loss = criterion(output, velocity_gt)
    epoch_loss += loss/len(train_loader)

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

  losses.append(epoch_loss.item())
  print("Epoch", epoch, "Loss:", epoch_loss.item())

  with torch.no_grad():
    val_loss = 0
    for batch in test_loader:
        # Forward pass
        velocity_gt, _ = batch
        output = autoencoder(velocity_gt)

        # Compute the loss
        val_loss += criterion(output, velocity_gt)/len(test_loader)
    
    val_losses.append(val_loss.item())
    print("Val Loss:", val_loss.item())


torch.save(autoencoder.state_dict(), f"./pretrained_models/autoencoder_{z_size}.pth")


plt.plot(losses, label="Train")
plt.plot(val_losses, label="Valid")
plt.legend()
plt.show()

# Symmetry-aware Autoencoder (SA-nCAE)

In [ ]:
from symmetry_aware_autoencoder import DiscreteSymmetries, SymAutoencoder, STN

z_size = 8
stn = STN((2,32,32))
discrete_symmetries = DiscreteSymmetries().to(device=device)
sae_autoencoder = Autoencoder(latent_dim=z_size, base=32, activation=nn.Tanh()).to(device=device)
sae_finalautoencoder = SymAutoencoder(sae_autoencoder, stn, discrete_symmetries).to(device=device)

pytorch_total_params = sum(p.numel() for p in sae_finalautoencoder.parameters() if p.requires_grad)
print(f"Number of trainable parameters in the autoencoder: {pytorch_total_params:,}")


def symmetry_criterion(input, output):
    loss = torch.mean((output - input)**2, dim=(-3, -2, -1))
    loss = torch.min(loss, dim=1)
    loss = torch.mean(loss.values)
    return loss

criterion = symmetry_criterion
optimizer = optim.Adam(sae_finalautoencoder.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 30
pbar = tqdm(range(num_epochs))

losses = []
val_losses = []

for epoch in pbar:
  epoch_loss = 0
  for batch in train_loader:
    # Zero the gradients
    optimizer.zero_grad()
    velocity_gt, _ = batch
    
    # Forward pass
    input_sym, output_sym, _ = sae_finalautoencoder(velocity_gt)
    
    # get training loss
    loss = symmetry_criterion(input_sym, output_sym)

    epoch_loss += loss/len(train_loader)

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

  losses.append(epoch_loss.item())
  print("Epoch", epoch, "Loss:", epoch_loss.item())

  with torch.no_grad():
    val_loss = 0
    for batch in test_loader:
        # Forward pass
        velocity_gt, _ = batch

        input_sym, output_sym, sx = sae_finalautoencoder(velocity_gt)

        # get reconstruction loss
        min_sym = torch.argmin(torch.mean((output_sym - input_sym)**2, dim=(-3, -2, -1)), dim=1) # lowest loss

        output = []
        for i, s in enumerate(min_sym):
            a = sae_finalautoencoder.stn.shift_x_operator(output_sym[i, s], -1*sx[i,s].reshape(-1))
            a = sae_finalautoencoder.discrete_symmetries.inverse(a, s)
            output.append(a)

        output = torch.concatenate(output, dim=0)

        # Compute the loss
        val_loss += torch.mean((velocity_gt-output)**2)/len(test_loader)
    
    val_losses.append(val_loss.item())
    print("Val Loss:", val_loss.item())


torch.save(sae_finalautoencoder.state_dict(), f"./pretrained_models/sae_autoencoder_{z_size}.pth")

plt.plot(losses, label="Train")
plt.plot(val_losses, label="Valid")
plt.legend()
plt.show()